# Phase III Part A: Faber Baseline Strategy
Implements Faber's 10-month SMA momentum rule:
- **Buy** when month-end price > 10-month SMA
- **Move to cash** when month-end price < 10-month SMA
- Equal weight across all invested ETFs
- Monthly rebalancing from February 2007 to December 2024

In [1]:
import pandas as pd
import numpy as np

ETFS           = ['SPY', 'EFA', 'IEF', 'VNQ', 'DBC']
BACKTEST_START = '2010-09-30'
BACKTEST_END   = '2024-08-31'

prices = pd.read_csv('../cleaned_prices.csv', index_col=0, parse_dates=True)
macro  = pd.read_csv('../cleaned_macro.csv',  index_col=0, parse_dates=True)
#print(f'{len(prices)} month-end observations.')
#print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')

## Step 1: Calculate 10-Month SMA
For each ETF at each month-end, compute the simple moving average of the last 10 monthly closing prices.

In [2]:
# rolling(10) uses only past prices â€” no lookahead bias
sma_10 = prices[ETFS].rolling(10).mean()

print('10-month SMA calculated.')
print(f'First valid SMA date: {sma_10.dropna().index[0].date()}')

10-month SMA calculated.
First valid SMA date: 2006-11-30


## Step 2: Generate Buy/Sell Signals
Signal = 1 (invest) if price > SMA, else 0 (cash).

In [3]:
# Shift signals by 1 month: signal at end of month t-1 determines position during month t
signals = (prices[ETFS] > sma_10).astype(int).shift(1)

# Restrict to backtest period after shifting (so Sep 2010 uses Aug 2010 signal)
signals = signals.loc[BACKTEST_START:BACKTEST_END].fillna(0).astype(int)
prices_bt = prices[ETFS].loc[BACKTEST_START:BACKTEST_END]

signals.head()

,SPY,EFA,IEF,VNQ,DBC
2010-09-30,0,0,1,1,0
2010-10-31,1,1,1,1,1
2010-11-30,1,1,1,1,1
2010-12-31,1,1,1,1,1
2011-01-31,1,1,0,1,1


## Step 3: Run Monthly Rebalancing Loop
At each month-end:
- Identify which ETFs have price > 10-month SMA
- Assign equal weight to each invested ETF
- If no ETFs qualify, portfolio sits in cash (0% return)
- Record the weighted portfolio return for that month

In [4]:
# TB3MS is annualised %, convert to monthly decimal
tb3ms_monthly = (macro["TB3MS"] / 100 / 12).loc[BACKTEST_START:BACKTEST_END]

monthly_returns = prices[ETFS].pct_change().loc[BACKTEST_START:BACKTEST_END]

WEIGHT = 0.2   # fixed 20% per ETF per Faber (2006)
COST   = 0.001 # 0.1% per unit of one-way turnover

prev_invested = pd.Series(0, index=ETFS)
portfolio_returns = []

for date in signals.index:
    invested  = signals.loc[date]
    cash_rate = tb3ms_monthly.get(date, 0.0)

    turnover = (invested - prev_invested).abs().sum() * WEIGHT
    txn_cost = COST * turnover

    etf_ret  = (monthly_returns.loc[date] * invested       * WEIGHT).sum()
    cash_ret = ((1 - invested)             * WEIGHT * cash_rate).sum()

    portfolio_returns.append({"date": date, "return": etf_ret + cash_ret - txn_cost,
                               "n_invested": int(invested.sum())})
    prev_invested = invested

faber_returns = pd.DataFrame(portfolio_returns).set_index("date")

faber_returns.head(10)

,return,n_invested
date,,
2010-09-30,0.008610,2
2010-10-31,0.032999,5
2010-11-30,-0.016020,5
2010-12-31,0.052011,5
2011-01-31,0.022292,4
2011-02-28,0.031821,4
2011-03-31,-0.002641,4
2011-04-30,0.037659,4
2011-05-31,-0.014244,4


## Step 4: Save Results

In [5]:
faber_returns.to_csv('../faber_returns.csv')